In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torchvision.transforms import v2
from torch.utils.data import DataLoader, random_split
from tqdm.notebook import tqdm
from PIL import ImageFile

# Prevent crashes from slightly corrupted images
ImageFile.LOAD_TRUNCATED_IMAGES = True

# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

IMG_SIZE = 300 
BATCH_SIZE = 32 
DATA_PATH = '/kaggle/input/datasets/shruthisindhura/pestopia/Datasets/Pest_Dataset'

# ==========================================
# 2. ADVANCED AUGMENTATION & DATA LOADING
# ==========================================
def get_dataloaders():
    train_tfms = transforms.Compose([
        transforms.Resize((320, 320)),
        transforms.RandomCrop((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(30), 
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_tfms = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    full_ds = datasets.ImageFolder(DATA_PATH)
    train_sz = int(0.8 * len(full_ds))
    val_sz = len(full_ds) - train_sz
    
    generator = torch.Generator().manual_seed(42)
    train_idx, val_idx = random_split(range(len(full_ds)), [train_sz, val_sz], generator=generator)
    
    train_ds = datasets.ImageFolder(DATA_PATH, transform=train_tfms)
    val_ds = datasets.ImageFolder(DATA_PATH, transform=val_tfms)
    
    train_sub = torch.utils.data.Subset(train_ds, train_idx.indices)
    val_sub = torch.utils.data.Subset(val_ds, val_idx.indices)
    
    train_loader = DataLoader(train_sub, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_sub, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    
    return train_loader, val_loader, len(full_ds.classes)

train_loader, val_loader, NUM_CLASSES = get_dataloaders()
print(f"✅ Data Setup Complete. Classes: {NUM_CLASSES}")

# ==========================================
# 3. EFFICIENTNET-B3 MODEL SETUP
# ==========================================
model = models.efficientnet_b3(weights='DEFAULT')

num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Sequential(
    nn.Dropout(p=0.5, inplace=True), # High dropout to stop memorization
    nn.Linear(num_ftrs, NUM_CLASSES)
)
model = model.to(DEVICE)

# ==========================================
# 4. OPTIMIZER, SCHEDULER & MIXUP SETUP
# ==========================================
# We use Standard CrossEntropy with Label Smoothing since MixUp handles the imbalance heavily
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# AdamW with heavy weight decay
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-3)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-6)

# The MixUp / CutMix blending tool
mixup_cutmix = v2.RandomChoice([
    v2.MixUp(num_classes=NUM_CLASSES, alpha=0.2),
    v2.CutMix(num_classes=NUM_CLASSES, alpha=1.0)
])

# ==========================================
# 5. TRAINING LOOP
# ==========================================
def run_epoch(loader, is_train):
    model.train() if is_train else model.eval()
    total_loss, correct = 0.0, 0
    iterator = tqdm(loader, desc="Training" if is_train else "Validating", leave=False)
    
    with torch.set_grad_enabled(is_train):
        for img, label in iterator:
            img, label = img.to(DEVICE), label.to(DEVICE)
            
            if is_train:
                # Apply MixUp blending (forces the model to generalize)
                label_one_hot = torch.nn.functional.one_hot(label, num_classes=NUM_CLASSES).float()
                img, blended_labels = mixup_cutmix(img, label_one_hot)
                
                optimizer.zero_grad()
                out = model(img)
                loss = criterion(out, blended_labels)
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item() * img.size(0)
            else:
                out = model(img)
                loss = criterion(out, label)
                total_loss += loss.item() * img.size(0)
                correct += (out.argmax(1) == label).sum().item()
                
    # We do not compute standard "Train Accuracy" because MixUp images are blended and unreadable as a single class
    val_acc = 0 if is_train else (correct / len(loader.dataset))
    return total_loss / len(loader.dataset), val_acc

# ==========================================
# 6. EXECUTION 
# ==========================================
EPOCHS = 40
best_acc = 0.0
patience, patience_counter = 7, 0 

print(f"\n🚀 Starting Training for up to {EPOCHS} Epochs...")
for epoch in range(EPOCHS):
    t_loss, _ = run_epoch(train_loader, True) # Train Accuracy ignored due to Mixup
    v_loss, v_acc = run_epoch(val_loader, False)
    
    scheduler.step(v_loss)
    current_lr = optimizer.param_groups[0]['lr']
    
    print(f"Ep {epoch+1:02d}/{EPOCHS} | LR: {current_lr:.6f} | Train Loss: {t_loss:.4f} | Val Acc: {v_acc*100:.2f}% | Val Loss: {v_loss:.4f}")
    
    if v_acc > best_acc:
        best_acc = v_acc
        torch.save(model.state_dict(), 'efficientnet_b3_pestopia_ultimate.pth')
        print(f"   🏆 New Best Validation Score Saved! ({best_acc*100:.2f}%)")
        patience_counter = 0
    else:
        patience_counter += 1
        
    if patience_counter >= patience:
        print(f"\n🛑 Early stopping triggered. Validation accuracy plateaued for {patience} epochs.")
        break

print(f"\n✅ Training Complete. Best Validation Accuracy Achieved: {best_acc*100:.2f}%")

Using device: cuda
✅ Data Setup Complete. Classes: 132
Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 197MB/s]



🚀 Starting Training for up to 40 Epochs...


Training:   0%|          | 0/1398 [00:00<?, ?it/s]

Validating:   0%|          | 0/350 [00:00<?, ?it/s]

...

Validating:   0%|          | 0/350 [00:00<?, ?it/s]

Ep 39/40 | LR: 0.000250 | Train Loss: 1.8044 | Val Acc: 74.68% | Val Loss: 1.7274


Training:   0%|          | 0/1398 [00:00<?, ?it/s]

Validating:   0%|          | 0/350 [00:00<?, ?it/s]

Ep 40/40 | LR: 0.000250 | Train Loss: 1.7674 | Val Acc: 75.45% | Val Loss: 1.6920
   🏆 New Best Validation Score Saved! (75.45%)

✅ Training Complete. Best Validation Accuracy Achieved: 75.45%
